In [1]:
import kagglehub
benjaminkz_places365_path = kagglehub.dataset_download('benjaminkz/places365')


In [4]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.init as init
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report

# --------------------------- Configuration --------------------------- #
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# benjaminkz_places365_path = "/path/to/places365"  # CHANGE THIS

train_txt = os.path.join(benjaminkz_places365_path, 'train.txt')
print(train_txt)
val_txt = os.path.join(benjaminkz_places365_path, 'val.txt')
train_dir = os.path.join(benjaminkz_places365_path, 'train')
val_dir = os.path.join(benjaminkz_places365_path, 'val')

# --------------------------- Dataset Class --------------------------- #
class Places365Dataset(Dataset):
    def __init__(self, txt_file, root_dir, transform=None):
        self.samples = []
        self.classes = set()

        with open(txt_file, 'r') as f:
            for line in f:
                rel_path = line.strip()
                full_path = os.path.join(root_dir, rel_path)
                class_name = rel_path.split('/')[1]  # e.g., train/raceway/img.jpg → raceway
                self.samples.append((full_path, class_name))
                self.classes.add(class_name)

        self.classes = sorted(list(self.classes))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        self.samples = [(path, self.class_to_idx[label]) for path, label in self.samples]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# --------------------------- Transforms --------------------------- #
trainTransform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

testTransform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# --------------------------- Datasets and Loaders --------------------------- #
# train_full_dataset = Places365Dataset(train_txt, benjaminkz_places365_path, transform=None)
train_full_dataset = Places365Dataset(train_txt, benjaminkz_places365_path, transform=None)
class_names = train_full_dataset.classes  # This gives you the list of class names

train_size = int(0.85 * len(train_full_dataset))
val_size = len(train_full_dataset) - train_size
train_dataset, val_dataset = random_split(train_full_dataset, [train_size, val_size])

# Apply transforms after splitting
train_dataset.dataset.transform = trainTransform
val_dataset.dataset.transform = testTransform

test_dataset = Places365Dataset(val_txt, benjaminkz_places365_path, transform=testTransform)

batchSize = 64
train_loader = DataLoader(train_dataset, batch_size=batchSize, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batchSize, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batchSize, shuffle=False)

# --------------------------- ResNeXt Model --------------------------- #
class ResNeXtBlock(nn.Module):
    expansion = 2

    def __init__(self, in_channels, out_channels, stride=1, cardinality=32, downsample=None):
        super(ResNeXtBlock, self).__init__()
        D = int(out_channels / self.expansion)
        group_width = int(D / cardinality)

        self.conv1 = nn.Conv2d(in_channels, D, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(D)
        self.conv2 = nn.Conv2d(D, D, kernel_size=3, stride=stride, padding=1, groups=cardinality, bias=False)
        self.bn2 = nn.BatchNorm2d(D)
        self.conv3 = nn.Conv2d(D, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.relu(out)
        return out

class ResNeXt(nn.Module):
    def __init__(self, num_classes=365, cardinality=32, init_method='he'):
        super(ResNeXt, self).__init__()
        self.in_channels = 64
        self.cardinality = cardinality

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(128, 3)
        self.layer2 = self._make_layer(256, 4, stride=2)
        self.layer3 = self._make_layer(512, 6, stride=2)
        self.layer4 = self._make_layer(1024, 3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(1024, num_classes)

        self._initialize_weights(init_method)

    def _make_layer(self, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        layers = [ResNeXtBlock(self.in_channels, out_channels, stride, self.cardinality, downsample)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(ResNeXtBlock(self.in_channels, out_channels, cardinality=self.cardinality))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

    def _initialize_weights(self, method='he'):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                if method == 'xavier':
                    init.xavier_normal_(m.weight)
                elif method == 'he':
                    init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                init.constant_(m.weight, 1)
                init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                init.normal_(m.weight, 0, 0.01)
                init.constant_(m.bias, 0)

# --------------------------- Training Utilities --------------------------- #
def evaluate_model(model, loader, criterion, device='cuda'):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    loss = running_loss / len(loader.dataset)
    acc = correct / total
    return loss, acc, all_preds, all_labels

def train_model(model, train_loader, val_loader, criterion, optimizer, model_name_prefix, scheduler=None, num_epochs=25, device='cuda'):
    os.makedirs("saved_models", exist_ok=True)
    model = model.to(device)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'best_acc': 0}
    best_acc = 0.0

    for epoch in tqdm(range(num_epochs)):
        start_time = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = correct / total
        val_loss, val_acc, _, _ = evaluate_model(model, val_loader, criterion, device)

        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_acc)
            else:
                scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            history['best_acc'] = best_acc
            torch.save(model.state_dict(), f"saved_models/{model_name_prefix}.pth")

        print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
    return history

def plot_history(history):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.legend()
    plt.show()

def plot_confusion_matrix(cm, classes):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

# --------------------------- Run Training --------------------------- #
num_classes = 365
model = ResNeXt(num_classes=num_classes, init_method='he', cardinality=32).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

model_name_prefix = "resnext_places365"
history = train_model(model, train_loader, val_loader, criterion, optimizer, model_name_prefix, scheduler, num_epochs=3, device=device)

# --------------------------- Evaluation --------------------------- #
test_loss, test_acc, test_preds, test_labels = evaluate_model(model, test_loader, criterion, device)
cm = confusion_matrix(test_labels, test_preds)
plot_history(history)
# plot_confusion_matrix(cm, classes=[str(i) for i in range(num_classes)])
plot_confusion_matrix(cm, classes=class_names)
print("Classification Report:")
# print(classification_report(test_labels, test_preds, digits=3))
print(classification_report(test_labels, test_preds, target_names=class_names, digits=3))

print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")


/Users/vishali/.cache/kagglehub/datasets/benjaminkz/places365/versions/1/train.txt


  0%|          | 0/15 [5:24:46<?, ?it/s]


KeyboardInterrupt: 